# Walkthrough

This notebook goes through the whole analysis. It computes the anomaly statistics of the Planck maps, makes an ensemble of LCDM realizations, and compares the two. It needs the maps and the mask described in `data/README.md`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np

import maps, simulate, analyze, plots, stats

RESULTS = os.path.join(maps.HERE, 'results')
os.makedirs(os.path.join(RESULTS, 'figures'), exist_ok=True)

# the multipole ranges of S_1/2 and R^TT, used for the data and the realizations alike
S12_LMAX = 40
RTT_LMAX = 27
NSIM = 10_000

## 1. Statistics of the Planck maps

Each map and the common mask are rotated to ecliptic coordinates and downgraded to $N_{\rm side} = 64$ and $N_{\rm side} = 16$.
- $S_{1/2}$ uses the masked $N_{\rm side} = 64$ map, with pseudo-$C_\ell$ deconvolved by NaMaster.
- $R^{TT}$ uses the full $N_{\rm side} = 64$ sky.
- $\sigma^2_{16}$ is the pixel variance of the masked northern ecliptic sky at $N_{\rm side} = 16$.

$S_{QO}$ and $T_{QO}$ of the data come from Copi's multipole vector code. They start as nan, so fill them in by hand before comparing. Computing the statistics reads about 7 GB of maps.

In [ ]:
mask64, mask16 = maps.prepare_mask()
bins, workspace = maps.s12_workspace(mask64)
data = {n: maps.data_statistics(n, mask64, mask16, bins, workspace, s12_lmax=S12_LMAX, rtt_lmax=RTT_LMAX)
        for n in maps.MAP_FILES if os.path.exists(os.path.join(maps.DATA_DIR, maps.MAP_FILES[n]))}

# data['smica'][3:] = S_QO, T_QO

for name, d in data.items():
    print(f'{name:>10}', '  '.join(f'{s} = {v:.4g}' for s, v in zip(analyze.STATS, d)))

## 2. Realizations

`simulate.run` draws $a_{\ell m}$ from the Planck best-fit spectrum up to $\ell = 200$. It makes each map at the resolution its statistic needs, with the matching beam and pixel window, and applies the same mask as the data. Each realization takes about 15 ms. Without `mpd_decomp.py` on the path, the $S_{QO}$ and $T_{QO}$ columns are nan.

For a larger ensemble, run `cluster/run_sims.sbatch`, or run `python simulate.py` several times with different seeds. `analyze.load_sims` stacks every file in `results/sims/`.

In [ ]:
s = simulate.setup((mask64, mask16), s12_lmax=S12_LMAX, rtt_lmax=RTT_LMAX)
os.makedirs(os.path.join(RESULTS, 'sims'), exist_ok=True)
np.save(os.path.join(RESULTS, 'sims', 'sims_notebook.npy'), simulate.run(NSIM, seed=12345, s=s))
sims = analyze.load_sims(os.path.join(RESULTS, 'sims', '*.npy'))

## 3. p-values

A realization counts as anomalous when it is at least as extreme as the data in the same direction: lower $S_{1/2}$, $R^{TT}$ and $\sigma^2_{16}$, and higher $S_{QO}$.
- The diagonal of each table holds the individual $p$-values.
- Below the diagonal are the joint pairwise $p$-values.
- Above the diagonal is the ratio of each joint $p$-value to the product of the two individual ones.

Statistics missing from the data or the realizations are skipped.

In [ ]:
for name, d in data.items():
    j = analyze.joint_summary(sims, d)
    print(f"{name}: joint p of {', '.join(j['stats'])} = {j['p']:.2g} from {j['n']} realizations")
    print(analyze.format_table(analyze.pairwise_table(sims, d)), end='\n\n')

## 4. Figures

The first figure shows the distribution of each statistic in the realizations, with a line for each map. The second figure shows each pair of statistics, with the maps marked. Both are saved in `results/figures/` as PDF and PNG.

In [ ]:
fig = plots.histograms(sims, data, os.path.join(RESULTS, 'figures', 'fig_histograms'))

In [ ]:
fig = plots.pairs(sims, data, os.path.join(RESULTS, 'figures', 'fig_pairs'))